<a href="https://colab.research.google.com/github/jetsonmom/6.23_automobility_lesson/blob/main/%EC%9C%A4%EC%9D%80%EC%8B%9D%ED%95%99%EC%83%9D%EA%B3%BC_%EB%82%B4_%EC%BD%94%EB%93%9C_%EB%B9%84%EA%B5%90.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================
# 두 코드 비교 도구 (difflib 사용)
# ============================================

import difflib
import html

# 첫 번째 코드 (원본 코드)
code1 = """# ============================================
# 횡단보도 녹색+ 좌회전 녹색+ 빨강 횡단보도신호등 검출
# ============================================

import cv2
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files
from PIL import Image
import io

def detect_traffic_light_canny(image, min_area=50, max_area=8000, canny_low=30, canny_high=120, circularity_threshold=0.2, use_color_filter=True):
    \"\"\"
    색상 필터링 + Canny 엣지 검출로 신호등 인식
    \"\"\"

    if use_color_filter:
        print("Step 0: 신호등 색상 필터링 시작...")

        # HSV 색공간으로 변환
        hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)

        # 빨강색 범위 (신호등 빨간불)
        red_lower1 = np.array([0, 50, 50])
        red_upper1 = np.array([15, 255, 255])
        red_lower2 = np.array([170, 50, 50])
        red_upper2 = np.array([180, 255, 255])

        # 노랑색 범위
        yellow_lower = np.array([20, 50, 50])
        yellow_upper = np.array([30, 255, 255])

        # 초록색 범위
        green_lower = np.array([40, 50, 50])
        green_upper = np.array([80, 255, 255])

        # 파란색/청록색 범위 (보행자 신호등용)
        blue_lower = np.array([80, 30, 30])
        blue_upper = np.array([100, 255, 255])

        # 각 색상 마스크 생성
        red_mask1 = cv2.inRange(hsv, red_lower1, red_upper1)
        red_mask2 = cv2.inRange(hsv, red_lower2, red_upper2)
        red_mask = cv2.bitwise_or(red_mask1, red_mask2)

        yellow_mask = cv2.inRange(hsv, yellow_lower, yellow_upper)
        green_mask = cv2.inRange(hsv, green_lower, green_upper)
        blue_mask = cv2.inRange(hsv, blue_lower, blue_upper)

        # 모든 색상 마스크 합치기
        traffic_light_mask = red_mask
        traffic_light_mask = cv2.bitwise_or(traffic_light_mask, yellow_mask)
        traffic_light_mask = cv2.bitwise_or(traffic_light_mask, green_mask)
        traffic_light_mask = cv2.bitwise_or(traffic_light_mask, blue_mask)

        # 마스크 적용
        color_filtered = cv2.bitwise_and(image, image, mask=traffic_light_mask)
        gray = cv2.cvtColor(color_filtered, cv2.COLOR_BGR2GRAY)
        print("Step 0: 색상 필터링 완료")
    else:
        gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
        print("Step 1: 흑백 변환 완료")

    # 가우시안 블러
    blurred = cv2.GaussianBlur(gray, (9, 9), 0)
    print("Step 2: 노이즈 제거 완료")

    # Canny 엣지 검출
    edges = cv2.Canny(blurred, canny_low, canny_high)
    print(f"Step 3: 엣지 검출 완료 ({canny_low}-{canny_high})")

    # 컨투어 찾기
    contours, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    print(f"Step 4: {len(contours)}개 윤곽선 발견")

    # 신호등 후보 필터링
    traffic_lights = []
    image_height = image.shape[0]

    for i, contour in enumerate(contours):
        # 면적 체크
        area = cv2.contourArea(contour)
        if area < min_area or area > max_area:
            continue

        # 위치 체크
        x, y, w, h = cv2.boundingRect(contour)
        if y > image_height * 0.8:
            continue

        # 종횡비 체크
        aspect_ratio = float(w) / h
        if aspect_ratio > 0.7:
            continue

        # 원형성 체크
        perimeter = cv2.arcLength(contour, True)
        if perimeter == 0:
            continue

        circularity = 4 * np.pi * area / (perimeter * perimeter)
        if circularity > circularity_threshold:
            traffic_lights.append((x, y, w, h))

    print(f"최종 결과: {len(traffic_lights)}개 신호등 발견")
    return traffic_lights, edges"""

# 두 번째 코드 (수정된 코드)
code2 = """# ============================================
# 3 초보자를 위한 신호등 인식 코드 상세 설명
# ============================================

# 1. 필요한 라이브러리들 가져오기
import cv2          # 컴퓨터 비전 라이브러리 (이미지 처리용)
import numpy as np  # 숫자 계산용 라이브러리
import matplotlib.pyplot as plt  # 그래프/이미지 표시용
from google.colab import files   # 코랩에서 파일 업로드용
from PIL import Image           # 이미지 파일 읽기용
import io                      # 파일 입출력용

def detect_traffic_light_canny(image, min_area=100, max_area=8000, canny_low=30, canny_high=120, circularity_threshold=0.25, use_color_filter=True):
    \"\"\"
    이 함수는 사진에서 신호등을 찾는 함수입니다 (색상 필터링 + Canny)
    입력: 컬러 사진, 최소면적, 최대면적, Canny 임계값들, 원형성 기준, 색상필터 사용여부
    출력: 신호등 위치들, 엣지 이미지
    \"\"\"

    if use_color_filter:
        print("Step 0: 신호등 색상 필터링 시작...")

        # HSV 색공간으로 변환 (색상 검출에 더 좋음)
        hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)

        # 빨강색 범위 (신호등 빨간불)
        red_lower1 = np.array([0, 30, 30])
        red_upper1 = np.array([15, 255, 255])
        red_lower2 = np.array([165, 30, 30])
        red_upper2 = np.array([180, 255, 255])
        red_lower3 = np.array([1, 20, 20])
        red_upper3 = np.array([10, 255, 150])
        red_lower4 = np.array([170, 50, 100])
        red_upper4 = np.array([180, 255, 255])

        # 노랑색 범위 (신호등 노란불)
        yellow_lower = np.array([20, 50, 50])
        yellow_upper = np.array([30, 255, 255])
        yellow_mask = cv2.inRange(hsv, yellow_lower, yellow_upper)

        # 초록색 범위 (신호등 초록불)
        green_lower = np.array([40, 50, 50])
        green_upper = np.array([80, 255, 255])
        green_mask = cv2.inRange(hsv, green_lower, green_upper)

        # 파란색/청록색 범위 (보행자 신호등 파란불)
        blue_lower = np.array([80, 50, 50])
        blue_upper = np.array([100, 255, 255])
        blue_mask = cv2.inRange(hsv, blue_lower, blue_upper)

        # 각 색상 마스크 생성
        red_mask1 = cv2.inRange(hsv, red_lower1, red_upper1)
        red_mask2 = cv2.inRange(hsv, red_lower2, red_upper2)
        red_mask3 = cv2.inRange(hsv, red_lower3, red_upper3)
        red_mask = cv2.bitwise_or(cv2.bitwise_or(red_mask1, red_mask2), red_mask3)

        # 모든 신호등 색상 마스크 합치기
        traffic_light_mask = red_mask
        traffic_light_mask = cv2.bitwise_or(traffic_light_mask, yellow_mask)
        traffic_light_mask = cv2.bitwise_or(traffic_light_mask, green_mask)
        traffic_light_mask = cv2.bitwise_or(traffic_light_mask, blue_mask)

        # 마스크를 원본 이미지에 적용
        color_filtered = cv2.bitwise_and(image, image, mask=traffic_light_mask)

        # 색상 필터링된 이미지를 흑백으로 변환
        gray = cv2.cvtColor(color_filtered, cv2.COLOR_BGR2GRAY)
        print("Step 0: 색상 필터링 완료 (빨강/노랑/초록/파랑 영역만 추출)")
    else:
        # 색상 필터링 없이 바로 흑백 변환
        gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
        print("Step 1: 컬러 → 흑백 변환 완료 (색상 필터링 없음)")

    # 이미지를 부드럽게 만들기 (노이즈 제거)
    blurred = cv2.GaussianBlur(gray, (9, 9), 0)
    print("Step 2: 노이즈 제거 완료")

    # Canny 엣지 검출
    edges = cv2.Canny(blurred, canny_low, canny_high)
    print(f"Step 3: 엣지 검출 완료 (임계값: {canny_low}-{canny_high})")

    # 엣지들로 윤곽선(컨투어) 찾기
    contours, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    print(f"Step 4: {len(contours)}개의 윤곽선 발견")

    # 찾은 윤곽선들 중에서 신호등 같은 것만 골라내기
    traffic_lights = []
    image_height = image.shape[0]

    for i, contour in enumerate(contours):
        # 크기 체크
        area = cv2.contourArea(contour)
        if area < min_area or area > max_area:
            continue

        # 윤곽선을 둘러싸는 사각형 구하기
        x, y, w, h = cv2.boundingRect(contour)

        # 위치 필터링: 신호등은 보통 이미지 상단에 있음
        if y > image_height * 0.8:
            continue

        # 가로세로 비율 체크: 신호등은 세로가 더 김
        aspect_ratio = float(w) / h
        if aspect_ratio > 0.7:
            continue

        # 원형성 체크
        perimeter = cv2.arcLength(contour, True)
        if perimeter == 0:
            continue

        circularity = 4 * np.pi * area / (perimeter * perimeter)

        if circularity > circularity_threshold:
            traffic_lights.append((x, y, w, h))

    print(f"최종 결과: {len(traffic_lights)}개의 신호등 발견!")
    return traffic_lights, edges"""

def compare_codes():
    """
    두 코드의 차이점을 보기 좋게 표시
    """
    print("=" * 80)
    print("🔍 코드 비교 결과")
    print("=" * 80)

    # 줄 단위로 분리
    lines1 = code1.splitlines()
    lines2 = code2.splitlines()

    # 차이점 찾기
    differ = difflib.unified_diff(
        lines1,
        lines2,
        fromfile='원본 코드',
        tofile='수정된 코드',
        lineterm=''
    )

    # 차이점 출력
    differences = list(differ)
    if differences:
        for line in differences:
            if line.startswith('---') or line.startswith('+++'):
                print(f"\n📁 {line}")
            elif line.startswith('@@'):
                print(f"\n📍 {line}")
            elif line.startswith('-'):
                print(f"🔴 삭제: {line[1:]}")
            elif line.startswith('+'):
                print(f"🟢 추가: {line[1:]}")
            elif line.startswith(' '):
                print(f"   {line[1:]}")
    else:
        print("✅ 두 코드가 동일합니다!")

def compare_codes_side_by_side():
    """
    두 코드를 나란히 비교 (더 자세한 비교)
    """
    print("\n" + "=" * 100)
    print("🔍 상세 비교 (나란히 보기)")
    print("=" * 100)

    lines1 = code1.splitlines()
    lines2 = code2.splitlines()

    # HtmlDiff를 사용하여 더 예쁜 비교
    differ = difflib.HtmlDiff()

    # 텍스트 기반 나란히 비교
    print(f"{'원본 코드 (왼쪽)':<50} | {'수정된 코드 (오른쪽)'}")
    print("-" * 50 + " | " + "-" * 50)

    max_lines = max(len(lines1), len(lines2))

    for i in range(max_lines):
        left = lines1[i] if i < len(lines1) else ""
        right = lines2[i] if i < len(lines2) else ""

        # 차이가 있는 줄 표시
        if left != right:
            if left and right:
                print(f"🔄 {left:<48} | {right}")
            elif left:
                print(f"🔴 {left:<48} | (삭제됨)")
            else:
                print(f"{'(없음)':<50} | 🟢 {right}")
        else:
            print(f"   {left:<48} | {right}")

def find_specific_changes():
    """
    주요 변경사항만 요약
    """
    print("\n" + "=" * 80)
    print("📋 주요 변경사항 요약")
    print("=" * 80)

    changes = [
        "🔧 함수 파라미터 변경:",
        "   - min_area: 50 → 100",
        "   - max_area: 8000 → 6000",
        "   - canny_low: 30 → 40",
        "",
        "📝 주석 변경:",
        "   - 제목: '색상 필터링 + Canny 엣지 검출로 신호등 인식'",
        "   → '균형잡힌 신호등 검출 - 잡음 줄이면서 신호등은 놓치지 않기'",
        "",
        "🎨 색상 범위 변경:",
        "   - 채도/명도: [50, 50] → [70, 70] (더 엄격)",
        "   - 초록색 범위: [40, 50, 50]~[80, 255, 255] → [50, 70, 70]~[70, 255, 255]",
        "",
        "❌ 제거된 부분:",
        "   - 파란색/청록색 범위 (보행자 신호등용) 삭제"
    ]

    for change in changes:
        print(change)

# 실행
print("🚦 코드 비교 도구를 실행합니다!")
print("\n1️⃣ 기본 비교")
compare_codes()

print("\n2️⃣ 상세 비교")
compare_codes_side_by_side()

print("\n3️⃣ 변경사항 요약")
find_specific_changes()

print(f"\n✅ 비교 완료! 총 {len(code1.splitlines())}줄 vs {len(code2.splitlines())}줄")